# SQL 13: interview drills

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-13-interview-drills.ipynb)

Practice SQL against a small company dataset. Each question gives the business rule, the relevant sample rows and the full expected output. Write your query in the exercise cell, then open the solution when you are ready to compare approaches.

The runtime is Spark SQL 4.0. Setup installs PySpark in the notebook. Temporary views are read-only: the duplicate-removal and salary-update exercises return the rows to keep rather than modifying a table. Window filtering uses a subquery so these examples do not depend on QUALIFY.

## Setup

Run the setup and shared-data cells once. The problem statements display the relevant columns of these same tables; all sample rows are included unless explicitly labelled otherwise. Timestamps use UTC.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("sql-interview-drills").master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

## Shared sample data

These temporary views contain employees, salary updates, customers, orders and events. Each question repeats the relevant input columns so you can read it without scrolling back to these definitions. `null` in a sample table means SQL NULL.

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW departments AS
SELECT * FROM VALUES
  (1, 'Engineering', 'New York'),
  (2, 'Sales', 'Chicago'),
  (3, 'Marketing', 'Austin'),
  (4, 'Support', 'Remote')
AS departments(dept_id, dept_name, location)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW employees AS
SELECT * FROM VALUES
  (1, 'Alice', 1, NULL, 150000, DATE '2018-01-15', 'alice@example.com'),
  (6, 'Frank', 2, NULL, 130000, DATE '2017-05-20', 'frank@example.com'),
  (10, 'Judy', 3, NULL, 105000, DATE '2019-11-01', 'judy@example.com'),
  (12, 'Leo', 4, NULL, 90000, DATE '2018-08-08', 'leo@example.com'),
  (7, 'Grace', 2, 6, 98000, DATE '2020-02-14', 'grace@example.com'),
  (8, 'Heidi', 2, 6, 87000, DATE '2021-07-01', 'heidi@example.com'),
  (9, 'Ivan', 2, 6, 87000, DATE '2022-03-15', 'ivan@example.com'),
  (11, 'Kevin', 3, 10, 88000, DATE '2021-04-01', 'dave@example.com'),
  (2, 'Bob', 1, 1, 120000, DATE '2019-03-01', 'bob@example.com'),
  (3, 'Carol', 1, 1, 120000, DATE '2020-06-15', 'carol@example.com'),
  (13, 'Mallory', 4, 12, 72000, DATE '2022-05-05', 'mallory@example.com'),
  (14, 'Nina', 4, 12, 68000, DATE '2023-02-01', 'heidi@example.com'),
  (4, 'Dave', 1, 2, 95000, DATE '2021-09-01', 'dave@example.com'),
  (5, 'Eve', 1, 2, 90000, DATE '2022-01-10', 'eve@example.com')
AS employees(emp_id, name, dept_id, manager_id, salary, hire_date, email)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW employee_updates AS
SELECT * FROM VALUES
  (1, 1, 1, 155000, DATE '2023-01-01'),
  (2, 1, 1, 165000, DATE '2023-06-01'),
  (3, 2, 1, 122000, DATE '2023-02-01'),
  (4, 2, 1, 128000, DATE '2023-05-01'),
  (5, 3, 1, 125000, DATE '2023-03-01'),
  (6, 6, 2, 135000, DATE '2023-01-15'),
  (7, 6, 2, 140000, DATE '2023-07-01'),
  (8, 7, 2, 101000, DATE '2023-04-01')
AS employee_updates(update_id, emp_id, dept_id, salary, updated_at)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp', 'North', DATE '2022-01-05'),
  (2, 'Beta LLC', 'North', DATE '2022-02-10'),
  (3, 'Cyan Inc', 'South', DATE '2022-03-15'),
  (4, 'Delta Co', 'South', DATE '2022-04-20'),
  (5, 'Echo Ltd', 'West', DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West', DATE '2022-06-30'),
  (7, 'Gamma SA', 'East', DATE '2023-01-10'),
  (8, 'Hotel BV', 'East', DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1, 1, 'completed', 100, DATE '2023-01-05'),
  (2, 1, 'completed', 200, DATE '2023-02-10'),
  (3, 1, 'completed', 150, DATE '2023-03-15'),
  (4, 2, 'completed', 300, DATE '2023-01-20'),
  (5, 2, 'cancelled', 250, DATE '2023-02-05'),
  (6, 2, 'completed', 400, DATE '2023-03-20'),
  (7, 3, 'completed', 120, DATE '2023-01-25'),
  (8, 3, 'completed', 180, DATE '2023-02-28'),
  (9, 4, 'pending', 220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

spark.sql("""
CREATE OR REPLACE TEMP VIEW events AS
SELECT * FROM VALUES
  (1, 1, TIMESTAMP '2023-01-01 09:00:00', 'login'),
  (2, 1, TIMESTAMP '2023-01-02 09:05:00', 'login'),
  (3, 1, TIMESTAMP '2023-01-03 08:55:00', 'login'),
  (4, 1, TIMESTAMP '2023-01-05 09:00:00', 'login'),
  (5, 1, TIMESTAMP '2023-01-06 09:10:00', 'login'),
  (6, 2, TIMESTAMP '2023-01-01 10:00:00', 'login'),
  (7, 2, TIMESTAMP '2023-01-02 10:00:00', 'login'),
  (8, 2, TIMESTAMP '2023-01-04 10:00:00', 'login'),
  (9, 3, TIMESTAMP '2023-02-01 10:00:00', 'view'),
  (10, 3, TIMESTAMP '2023-02-01 10:05:00', 'click'),
  (11, 3, TIMESTAMP '2023-02-01 10:25:00', 'view'),
  (12, 3, TIMESTAMP '2023-02-01 11:30:00', 'purchase'),
  (13, 3, TIMESTAMP '2023-02-01 11:45:00', 'view'),
  (14, 4, TIMESTAMP '2023-02-01 10:00:00', 'login'),
  (15, 4, TIMESTAMP '2023-02-01 10:20:00', 'view')
AS events(event_id, user_id, event_ts, event_type)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW tiny_orders AS
SELECT * FROM VALUES
  (1),
  (NULL)
AS tiny_orders(customer_id)
""")

## Part A - Ranking and top-N

Most "top N per group" questions come down to choosing between `ROW_NUMBER`, `RANK` and `DENSE_RANK`, which differ
only in how they treat ties. If you need a refresher on partitions, peers and the three functions, read
[SQL 8: window functions](sql-08-window-functions.ipynb), sections 2 (ranking) and 7 (filtering on a window result),
before you start.

### Q1. Second-highest salary

**Difficulty:** Easy · about 5 min

Find the second-highest distinct salary across the company. Return one column named `second_highest`. Repeated salaries occupy one position; if no second value exists, return null.

**Sample input**

**`employees`**

| emp_id | name | salary |
| --- | --- | --- |
| 1 | Alice | 150000 |
| 6 | Frank | 130000 |
| 10 | Judy | 105000 |
| 12 | Leo | 90000 |
| 7 | Grace | 98000 |
| 8 | Heidi | 87000 |
| 9 | Ivan | 87000 |
| 11 | Kevin | 88000 |
| 2 | Bob | 120000 |
| 3 | Carol | 120000 |
| 13 | Mallory | 72000 |
| 14 | Nina | 68000 |
| 4 | Dave | 95000 |
| 5 | Eve | 90000 |

**Expected output**

| second_highest |
| --- |
| 130000 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT MAX(salary) AS second_highest
    FROM employees
    WHERE salary < (SELECT MAX(salary) FROM employees)
    """
spark.sql(q).show()
q = """
    SELECT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM employees
    )
    WHERE rnk = 2
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `MAX` below the overall maximum is the runner-up, because `<` excludes every copy of the top salary.
`DENSE_RANK() = 2` gives the same answer, because `DENSE_RANK` counts distinct values.

The trap is `ORDER BY salary DESC LIMIT 1 OFFSET 1`, which skips one *row*, not one *value*. When the top salary is
tied, the second row is another copy of the top salary. For salaries 150, 150 and 130, skipping one row returns 150; skipping one distinct salary returns 130.

</details>

### Q2. Nth-highest salary

**Difficulty:** Easy · about 5 min

Find the third-highest distinct salary. Return one column named `salary` and one row for the value, even when several employees earn it. Count distinct amounts rather than employees.

**Sample input**

**`employees`**

| emp_id | name | salary |
| --- | --- | --- |
| 1 | Alice | 150000 |
| 6 | Frank | 130000 |
| 10 | Judy | 105000 |
| 12 | Leo | 90000 |
| 7 | Grace | 98000 |
| 8 | Heidi | 87000 |
| 9 | Ivan | 87000 |
| 11 | Kevin | 88000 |
| 2 | Bob | 120000 |
| 3 | Carol | 120000 |
| 13 | Mallory | 72000 |
| 14 | Nina | 68000 |
| 4 | Dave | 95000 |
| 5 | Eve | 90000 |

**Expected output**

| salary |
| --- |
| 120000 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = "SELECT DISTINCT salary FROM employees ORDER BY salary DESC LIMIT 1 OFFSET 2"
spark.sql(q).show()
q = """
    SELECT DISTINCT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM employees
    )
    WHERE rnk = 3
    """
spark.sql(q).show()
q = """
    SELECT salary FROM (
        SELECT salary, DENSE_RANK() OVER (ORDER BY salary DESC) AS rnk FROM employees
    )
    WHERE rnk = 3
    ORDER BY salary DESC
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `DENSE_RANK` gives one rank per distinct value, so a tie produces several rows at the same rank: the third
query shows Bob and Carol both at rank 3. `DISTINCT` collapses them to one row. `ROW_NUMBER` never emits ties, but
it has no stable definition of "Nth" when values repeat.

The `LIMIT ... OFFSET` form only works because of `DISTINCT`. Without it, `OFFSET` counts rows. The salaries in
descending order are 150000, 130000, 120000, 120000, 105000, so the fourth-highest distinct salary is 105000, while
the fourth row is 120000.

</details>

### Q3. Top-3 per department, and how each ranking function counts

**Difficulty:** Medium · about 10 min

Return up to three employees per department, ordered by salary descending and employee ID ascending to break ties. Return `dept_id`, `name` and `salary`. Also report how many rows each department retains when selecting the first three positions with ROW_NUMBER, RANK and DENSE_RANK; the two rank functions must treat equal salaries as peers.

**Sample input**

**`employees`**

| emp_id | name | dept_id | salary |
| --- | --- | --- | --- |
| 1 | Alice | 1 | 150000 |
| 6 | Frank | 2 | 130000 |
| 10 | Judy | 3 | 105000 |
| 12 | Leo | 4 | 90000 |
| 7 | Grace | 2 | 98000 |
| 8 | Heidi | 2 | 87000 |
| 9 | Ivan | 2 | 87000 |
| 11 | Kevin | 3 | 88000 |
| 2 | Bob | 1 | 120000 |
| 3 | Carol | 1 | 120000 |
| 13 | Mallory | 4 | 72000 |
| 14 | Nina | 4 | 68000 |
| 4 | Dave | 1 | 95000 |
| 5 | Eve | 1 | 90000 |

**Expected output**

**Top employees**

| dept_id | name | salary |
| --- | --- | --- |
| 1 | Alice | 150000 |
| 1 | Bob | 120000 |
| 1 | Carol | 120000 |
| 2 | Frank | 130000 |
| 2 | Grace | 98000 |
| 2 | Heidi | 87000 |
| 3 | Judy | 105000 |
| 3 | Kevin | 88000 |
| 4 | Leo | 90000 |
| 4 | Mallory | 72000 |
| 4 | Nina | 68000 |

**Rows retained by each ranking function**

| dept_id | by_row_number | by_rank | by_dense_rank |
| --- | --- | --- | --- |
| 1 | 3 | 3 | 4 |
| 2 | 3 | 4 | 4 |
| 3 | 2 | 2 | 2 |
| 4 | 3 | 3 | 3 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT dept_id, name, salary FROM (
        SELECT dept_id, name, salary,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn
        FROM employees
    )
    WHERE rn <= 3
    ORDER BY dept_id, salary DESC, name
    """
spark.sql(q).show()
q = """
    SELECT dept_id,
           COUNT(CASE WHEN rn   <= 3 THEN 1 END) AS by_row_number,
           COUNT(CASE WHEN rnk  <= 3 THEN 1 END) AS by_rank,
           COUNT(CASE WHEN drnk <= 3 THEN 1 END) AS by_dense_rank
    FROM (
        SELECT dept_id, salary, emp_id,
               ROW_NUMBER() OVER (PARTITION BY dept_id ORDER BY salary DESC, emp_id) AS rn,
               RANK()       OVER (PARTITION BY dept_id ORDER BY salary DESC)         AS rnk,
               DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC)         AS drnk
        FROM employees
    )
    GROUP BY dept_id
    ORDER BY dept_id
    """
spark.sql(q).show()
q = """
    SELECT name, salary,
           RANK()       OVER by_salary AS rnk,
           DENSE_RANK() OVER by_salary AS drnk
    FROM employees
    WHERE dept_id = 1
    WINDOW by_salary AS (ORDER BY salary DESC)
    ORDER BY salary DESC, emp_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** Rank in an inner query and filter in the outer one, because a window cannot appear in `WHERE`. Use
`ROW_NUMBER` with a unique tie-breaker for "exactly N per group", and `RANK` or `DENSE_RANK` when tied rows should
all survive ([SQL 8](sql-08-window-functions.ipynb), section 7).

The counts table shows the difference. In department 1, Bob and Carol tie at 120000 for second place, so `RANK` keeps
three rows (1, 2, 2) while `DENSE_RANK` keeps four (1, 2, 2, 3, with Dave third). In department 2, Heidi and Ivan tie at
third place, so `RANK` and `DENSE_RANK` both keep four rows and `ROW_NUMBER` keeps three.

</details>

## Part B - Deduplication and history

These questions collapse many rows per key into one, safely and reproducibly. The technique (choosing the key, the
winning row and a deterministic tie-breaker) is taught in [SQL 9: deduplication](sql-09-deduplication.ipynb).

### Q4. Keep the latest row per employee

**Difficulty:** Medium · about 10 min

Return the latest update for each employee represented in `employee_updates`. Choose the greatest `updated_at`; if dates tie, choose the greatest `update_id`. Return `emp_id`, `dept_id`, `salary` and `updated_at`, ordered by employee ID. Employees without an update do not appear.

**Sample input**

**`employee_updates`**

| update_id | emp_id | dept_id | salary | updated_at |
| --- | --- | --- | --- | --- |
| 1 | 1 | 1 | 155000 | 2023-01-01 |
| 2 | 1 | 1 | 165000 | 2023-06-01 |
| 3 | 2 | 1 | 122000 | 2023-02-01 |
| 4 | 2 | 1 | 128000 | 2023-05-01 |
| 5 | 3 | 1 | 125000 | 2023-03-01 |
| 6 | 6 | 2 | 135000 | 2023-01-15 |
| 7 | 6 | 2 | 140000 | 2023-07-01 |
| 8 | 7 | 2 | 101000 | 2023-04-01 |

**Expected output**

| emp_id | dept_id | salary | updated_at |
| --- | --- | --- | --- |
| 1 | 1 | 165000 | 2023-06-01 |
| 2 | 1 | 128000 | 2023-05-01 |
| 3 | 1 | 125000 | 2023-03-01 |
| 6 | 2 | 140000 | 2023-07-01 |
| 7 | 2 | 101000 | 2023-04-01 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT emp_id, dept_id, salary, updated_at FROM (
        SELECT emp_id, dept_id, salary, updated_at,
               ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, update_id DESC) AS rn
        FROM employee_updates
    )
    WHERE rn = 1
    ORDER BY emp_id
    """
spark.sql(q).show()
q = """
    SELECT u.emp_id, u.dept_id, u.salary, u.updated_at
    FROM employee_updates u
    JOIN (
        SELECT emp_id, MAX(updated_at) AS max_ts FROM employee_updates GROUP BY emp_id
    ) m ON m.emp_id = u.emp_id AND m.max_ts = u.updated_at
    ORDER BY u.emp_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `ROW_NUMBER` ordered by the timestamp, with a unique tie-breaker, is the general recipe; Spark has no
`QUALIFY`, so `rn = 1` goes in an outer query. The `GROUP BY MAX` join returns both rows when two updates share the
latest timestamp, when there is a timestamp tie ([SQL 9](sql-09-deduplication.ipynb) explains this pitfall).

</details>

### Q5. Delete duplicates, keeping one row per employee

**Difficulty:** Medium · about 10 min

Return the update rows that would survive duplicate removal: one per employee, choosing the greatest `(updated_at, update_id)`. Return `update_id`, `emp_id` and `salary`, ordered by employee ID. This exercise selects survivors from read-only temporary views; it does not execute DELETE against a table.

**Sample input**

**`employee_updates`**

| update_id | emp_id | dept_id | salary | updated_at |
| --- | --- | --- | --- | --- |
| 1 | 1 | 1 | 155000 | 2023-01-01 |
| 2 | 1 | 1 | 165000 | 2023-06-01 |
| 3 | 2 | 1 | 122000 | 2023-02-01 |
| 4 | 2 | 1 | 128000 | 2023-05-01 |
| 5 | 3 | 1 | 125000 | 2023-03-01 |
| 6 | 6 | 2 | 135000 | 2023-01-15 |
| 7 | 6 | 2 | 140000 | 2023-07-01 |
| 8 | 7 | 2 | 101000 | 2023-04-01 |

**Expected output**

| update_id | emp_id | salary |
| --- | --- | --- |
| 2 | 1 | 165000 |
| 4 | 2 | 128000 |
| 5 | 3 | 125000 |
| 7 | 6 | 140000 |
| 8 | 7 | 101000 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT u.update_id, u.emp_id, u.salary
    FROM employee_updates u
    WHERE NOT EXISTS (
        SELECT 1 FROM employee_updates k
        WHERE k.emp_id = u.emp_id
          AND (k.updated_at > u.updated_at
               OR (k.updated_at = u.updated_at AND k.update_id > u.update_id))
    )
    ORDER BY u.emp_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** "Identify survivors, delete everything else" is the safe pattern. A row survives when no other row for the
same employee is newer, with `update_id` breaking a timestamp tie, which is exactly what the `NOT EXISTS` says. The
survivors match Q4. On a Delta or Iceberg table you would run `DELETE FROM ... WHERE update_id NOT IN (survivors)` or
a `MERGE`; the predicate is the same.

</details>

### Q6. Find duplicate emails

**Difficulty:** Easy · about 5 min

Find email addresses shared by more than one employee. Return `email`, `copies` and `people`, with names sorted alphabetically and joined with commas. Order the result by email. The supplied addresses are already normalized.

**Sample input**

**`employees`**

| emp_id | name | email |
| --- | --- | --- |
| 1 | Alice | alice@example.com |
| 6 | Frank | frank@example.com |
| 10 | Judy | judy@example.com |
| 12 | Leo | leo@example.com |
| 7 | Grace | grace@example.com |
| 8 | Heidi | heidi@example.com |
| 9 | Ivan | ivan@example.com |
| 11 | Kevin | dave@example.com |
| 2 | Bob | bob@example.com |
| 3 | Carol | carol@example.com |
| 13 | Mallory | mallory@example.com |
| 14 | Nina | heidi@example.com |
| 4 | Dave | dave@example.com |
| 5 | Eve | eve@example.com |

**Expected output**

| email | copies | people |
| --- | --- | --- |
| dave@example.com | 2 | Dave,Kevin |
| heidi@example.com | 2 | Heidi,Nina |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT email, COUNT(*) AS copies, array_join(sort_array(collect_list(name)), ',') AS people
    FROM employees
    GROUP BY email
    HAVING COUNT(*) > 1
    ORDER BY email
    """
spark.sql(q).show()
q = """
    SELECT emp_id, email FROM employees
    WHERE email IN (SELECT email FROM employees GROUP BY email HAVING COUNT(*) > 1)
    ORDER BY email, emp_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `GROUP BY email HAVING COUNT(*) > 1` is the whole answer. Real pipelines normalise the key first with
`lower(trim(email))`. `array_join(sort_array(collect_list(...)))` gives a deterministic list of names.

</details>

### Q7. Apply the latest salary update, leaving other employees alone

**Difficulty:** Medium · about 15 min

Produce a current salary view with one row per employee. Use the salary from the greatest `(updated_at, update_id)` when an update exists; otherwise keep the employee's original salary. Return `emp_id` and `salary`, ordered by employee ID. Do not modify the source views.

**Sample input**

**`employees`**

| emp_id | name | salary |
| --- | --- | --- |
| 1 | Alice | 150000 |
| 6 | Frank | 130000 |
| 10 | Judy | 105000 |
| 12 | Leo | 90000 |
| 7 | Grace | 98000 |
| 8 | Heidi | 87000 |
| 9 | Ivan | 87000 |
| 11 | Kevin | 88000 |
| 2 | Bob | 120000 |
| 3 | Carol | 120000 |
| 13 | Mallory | 72000 |
| 14 | Nina | 68000 |
| 4 | Dave | 95000 |
| 5 | Eve | 90000 |

**`employee_updates`**

| update_id | emp_id | dept_id | salary | updated_at |
| --- | --- | --- | --- | --- |
| 1 | 1 | 1 | 155000 | 2023-01-01 |
| 2 | 1 | 1 | 165000 | 2023-06-01 |
| 3 | 2 | 1 | 122000 | 2023-02-01 |
| 4 | 2 | 1 | 128000 | 2023-05-01 |
| 5 | 3 | 1 | 125000 | 2023-03-01 |
| 6 | 6 | 2 | 135000 | 2023-01-15 |
| 7 | 6 | 2 | 140000 | 2023-07-01 |
| 8 | 7 | 2 | 101000 | 2023-04-01 |

**Expected output**

| emp_id | salary |
| --- | --- |
| 1 | 165000 |
| 2 | 128000 |
| 3 | 125000 |
| 4 | 95000 |
| 5 | 90000 |
| 6 | 140000 |
| 7 | 101000 |
| 8 | 87000 |
| 9 | 87000 |
| 10 | 105000 |
| 11 | 88000 |
| 12 | 90000 |
| 13 | 72000 |
| 14 | 68000 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
CURRENT_SALARY_SQL = """
    WITH latest AS (
        SELECT emp_id, salary FROM (
            SELECT emp_id, salary,
                   ROW_NUMBER() OVER (PARTITION BY emp_id ORDER BY updated_at DESC, update_id DESC) AS rn
            FROM employee_updates
        )
        WHERE rn = 1
    )
    SELECT e.emp_id, COALESCE(l.salary, e.salary) AS salary
    FROM employees e
    LEFT JOIN latest l ON l.emp_id = e.emp_id
"""
q = CURRENT_SALARY_SQL + " ORDER BY emp_id"
spark.sql(q).show()
q = f"""
    SELECT COUNT(*) AS changed_rows
    FROM ({CURRENT_SALARY_SQL}) n
    JOIN employees o ON o.emp_id = n.emp_id AND o.salary <> n.salary
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** The `COALESCE` fallback is what keeps employees without an update from becoming `NULL`. A real table
would take the same result through `MERGE`, after the source has been deduplicated to one row per key.

</details>

## Part C - Running totals and period-over-period growth

Window aggregates keep every row while adding a cumulative or comparative value. Order the window by a unique key
when the ordering column has ties.

### Q8. Running total across all orders

**Difficulty:** Easy · about 5 min

For every order, return `order_id`, `order_date`, `amount` and a cumulative `running_total` across all customers. Include every status. Process rows by date, then order ID, so orders sharing a date still have a defined sequence.

**Sample input**

**`orders`**

| order_id | customer_id | status | amount | order_date |
| --- | --- | --- | --- | --- |
| 1 | 1 | completed | 100 | 2023-01-05 |
| 2 | 1 | completed | 200 | 2023-02-10 |
| 3 | 1 | completed | 150 | 2023-03-15 |
| 4 | 2 | completed | 300 | 2023-01-20 |
| 5 | 2 | cancelled | 250 | 2023-02-05 |
| 6 | 2 | completed | 400 | 2023-03-20 |
| 7 | 3 | completed | 120 | 2023-01-25 |
| 8 | 3 | completed | 180 | 2023-02-28 |
| 9 | 4 | pending | 220 | 2023-02-14 |
| 10 | 4 | completed | 320 | 2023-03-30 |
| 11 | 5 | completed | 500 | 2023-01-02 |
| 12 | 5 | completed | 450 | 2023-03-05 |
| 13 | 6 | completed | 275 | 2023-02-18 |
| 14 | 6 | cancelled | 325 | 2023-03-25 |

**Expected output**

| order_id | order_date | amount | running_total |
| --- | --- | --- | --- |
| 11 | 2023-01-02 | 500 | 500 |
| 1 | 2023-01-05 | 100 | 600 |
| 4 | 2023-01-20 | 300 | 900 |
| 7 | 2023-01-25 | 120 | 1020 |
| 5 | 2023-02-05 | 250 | 1270 |
| 2 | 2023-02-10 | 200 | 1470 |
| 9 | 2023-02-14 | 220 | 1690 |
| 13 | 2023-02-18 | 275 | 1965 |
| 8 | 2023-02-28 | 180 | 2145 |
| 12 | 2023-03-05 | 450 | 2595 |
| 3 | 2023-03-15 | 150 | 2745 |
| 6 | 2023-03-20 | 400 | 3145 |
| 14 | 2023-03-25 | 325 | 3470 |
| 10 | 2023-03-30 | 320 | 3790 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT order_id, order_date, amount,
           SUM(amount) OVER (ORDER BY order_date, order_id) AS running_total
    FROM orders
    ORDER BY order_date, order_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** A window's **frame** is the set of rows, within the partition, that the aggregate reads for the current
row. With `ORDER BY` and no explicit frame, the default is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`.
`RANGE` works on values, so all peers of the current row are inside the frame: rows that share an `ORDER BY` value
get the same total. Adding the unique `order_id` to the `ORDER BY` removes the ties, so the total advances row by row.

For two same-day orders of 10 and 20, ordering only by date gives both rows a total of 30. Adding order ID gives successive totals of 10 and 30.

</details>

### Q9. Running total per customer, completed orders only

**Difficulty:** Easy · about 5 min

Keep only completed orders, then calculate a running amount separately for each customer. Return `customer_id`, `order_date`, `amount` and `running_total`. Order each customer's rows by date and order ID. Cancelled and pending orders must not contribute to the total.

**Sample input**

**`orders`**

| order_id | customer_id | status | amount | order_date |
| --- | --- | --- | --- | --- |
| 1 | 1 | completed | 100 | 2023-01-05 |
| 2 | 1 | completed | 200 | 2023-02-10 |
| 3 | 1 | completed | 150 | 2023-03-15 |
| 4 | 2 | completed | 300 | 2023-01-20 |
| 5 | 2 | cancelled | 250 | 2023-02-05 |
| 6 | 2 | completed | 400 | 2023-03-20 |
| 7 | 3 | completed | 120 | 2023-01-25 |
| 8 | 3 | completed | 180 | 2023-02-28 |
| 9 | 4 | pending | 220 | 2023-02-14 |
| 10 | 4 | completed | 320 | 2023-03-30 |
| 11 | 5 | completed | 500 | 2023-01-02 |
| 12 | 5 | completed | 450 | 2023-03-05 |
| 13 | 6 | completed | 275 | 2023-02-18 |
| 14 | 6 | cancelled | 325 | 2023-03-25 |

**Expected output**

| customer_id | order_date | amount | running_total |
| --- | --- | --- | --- |
| 1 | 2023-01-05 | 100 | 100 |
| 1 | 2023-02-10 | 200 | 300 |
| 1 | 2023-03-15 | 150 | 450 |
| 2 | 2023-01-20 | 300 | 300 |
| 2 | 2023-03-20 | 400 | 700 |
| 3 | 2023-01-25 | 120 | 120 |
| 3 | 2023-02-28 | 180 | 300 |
| 4 | 2023-03-30 | 320 | 320 |
| 5 | 2023-01-02 | 500 | 500 |
| 5 | 2023-03-05 | 450 | 950 |
| 6 | 2023-02-18 | 275 | 275 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT customer_id, order_date, amount,
           SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date, order_id) AS running_total
    FROM orders
    WHERE status = 'completed'
    ORDER BY customer_id, order_date, order_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `WHERE` runs before the window, so the cancelled and pending orders never enter the sum. Filtering after
the window keeps their amounts in the running total. For customer 2, the cancelled order of 250 on 2023-02-05 would
push the March total to 950 instead of 700, when there is a timestamp tie.

</details>

### Q10. Month-over-month revenue growth

**Difficulty:** Medium · about 10 min

Sum completed-order revenue by calendar month. Return `month` as YYYY-MM, `revenue`, `prev_revenue` and `growth_pct`, ordered by month. Growth is 100 × (current − previous) / previous, rounded to two decimals. The first observed month's previous revenue and growth are null. All months in this sample are present and previous revenue is positive.

**Sample input**

**`orders`**

| order_id | customer_id | status | amount | order_date |
| --- | --- | --- | --- | --- |
| 1 | 1 | completed | 100 | 2023-01-05 |
| 2 | 1 | completed | 200 | 2023-02-10 |
| 3 | 1 | completed | 150 | 2023-03-15 |
| 4 | 2 | completed | 300 | 2023-01-20 |
| 5 | 2 | cancelled | 250 | 2023-02-05 |
| 6 | 2 | completed | 400 | 2023-03-20 |
| 7 | 3 | completed | 120 | 2023-01-25 |
| 8 | 3 | completed | 180 | 2023-02-28 |
| 9 | 4 | pending | 220 | 2023-02-14 |
| 10 | 4 | completed | 320 | 2023-03-30 |
| 11 | 5 | completed | 500 | 2023-01-02 |
| 12 | 5 | completed | 450 | 2023-03-05 |
| 13 | 6 | completed | 275 | 2023-02-18 |
| 14 | 6 | cancelled | 325 | 2023-03-25 |

**Expected output**

| month | revenue | prev_revenue | growth_pct |
| --- | --- | --- | --- |
| 2023-01 | 1020 | null | null |
| 2023-02 | 655 | 1020 | -35.78 |
| 2023-03 | 1320 | 655 | 101.53 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    WITH monthly AS (
        SELECT date_format(order_date, 'yyyy-MM') AS month, SUM(amount) AS revenue
        FROM orders
        WHERE status = 'completed'
        GROUP BY date_format(order_date, 'yyyy-MM')
    )
    SELECT month, revenue,
           LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
           ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
                 / LAG(revenue) OVER (ORDER BY month), 2) AS growth_pct
    FROM monthly
    ORDER BY month
    """
spark.sql(q).show()
q = """
    WITH monthly AS (
        SELECT date_format(order_date, 'yyyy-MM') AS month, SUM(amount) AS revenue
        FROM orders
        WHERE status = 'completed'
        GROUP BY date_format(order_date, 'yyyy-MM')
    )
    SELECT month, ROUND(revenue * 1.0 / LAG(revenue) OVER (ORDER BY month), 3) AS ratio
    FROM monthly
    ORDER BY month
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** Aggregate to one row per month first, then apply `LAG`. Applied to raw orders, `LAG` returns the previous
order. Keep the first period's growth `NULL` rather than `0`, so "no data" is not confused with "no growth".

</details>

### Q11. Each customer's share of completed revenue

**Difficulty:** Medium · about 10 min

For each customer with completed orders, return `customer_id`, completed revenue as `total`, and `pct_of_total`, rounded to two decimals. Use completed revenue across all customers as the denominator. Order by customer ID; customers with no completed orders are absent.

**Sample input**

**`orders`**

| order_id | customer_id | status | amount | order_date |
| --- | --- | --- | --- | --- |
| 1 | 1 | completed | 100 | 2023-01-05 |
| 2 | 1 | completed | 200 | 2023-02-10 |
| 3 | 1 | completed | 150 | 2023-03-15 |
| 4 | 2 | completed | 300 | 2023-01-20 |
| 5 | 2 | cancelled | 250 | 2023-02-05 |
| 6 | 2 | completed | 400 | 2023-03-20 |
| 7 | 3 | completed | 120 | 2023-01-25 |
| 8 | 3 | completed | 180 | 2023-02-28 |
| 9 | 4 | pending | 220 | 2023-02-14 |
| 10 | 4 | completed | 320 | 2023-03-30 |
| 11 | 5 | completed | 500 | 2023-01-02 |
| 12 | 5 | completed | 450 | 2023-03-05 |
| 13 | 6 | completed | 275 | 2023-02-18 |
| 14 | 6 | cancelled | 325 | 2023-03-25 |

**Expected output**

| customer_id | total | pct_of_total |
| --- | --- | --- |
| 1 | 450 | 15.03 |
| 2 | 700 | 23.37 |
| 3 | 300 | 10.02 |
| 4 | 320 | 10.68 |
| 5 | 950 | 31.72 |
| 6 | 275 | 9.18 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT customer_id, SUM(amount) AS total,
           ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct_of_total
    FROM orders
    WHERE status = 'completed'
    GROUP BY customer_id
    ORDER BY customer_id
    """
spark.sql(q).show()
q = """
    SELECT ROUND(SUM(pct), 2) FROM (
        SELECT ROUND(100.0 * SUM(amount) / SUM(SUM(amount)) OVER (), 2) AS pct
        FROM orders WHERE status = 'completed' GROUP BY customer_id
    )
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `SUM(SUM(amount)) OVER ()` nests a window over an aggregate. Multiply by `100.0` before dividing to avoid
integer division. Rounded shares are not guaranteed to total 100 in general, so check it.

</details>

## Part D - Joins and set logic

Self-joins, anti-joins and `LEFT JOIN` with an `IS NULL` test are the core of dimensional queries. The question
usually hides a `NULL` or an `EXISTS` versus `IN` decision.

### Q12. Each employee with their manager's name

**Difficulty:** Easy · about 5 min

Return every employee with their manager's name. Output `emp_id`, `name` and `manager`, ordered by employee ID. A null manager ID means the employee has no manager; keep that employee and show a null manager name.

**Sample input**

**`employees`**

| emp_id | name | manager_id |
| --- | --- | --- |
| 1 | Alice | null |
| 6 | Frank | null |
| 10 | Judy | null |
| 12 | Leo | null |
| 7 | Grace | 6 |
| 8 | Heidi | 6 |
| 9 | Ivan | 6 |
| 11 | Kevin | 10 |
| 2 | Bob | 1 |
| 3 | Carol | 1 |
| 13 | Mallory | 12 |
| 14 | Nina | 12 |
| 4 | Dave | 2 |
| 5 | Eve | 2 |

**Expected output**

| emp_id | name | manager |
| --- | --- | --- |
| 1 | Alice | null |
| 2 | Bob | Alice |
| 3 | Carol | Alice |
| 4 | Dave | Bob |
| 5 | Eve | Bob |
| 6 | Frank | null |
| 7 | Grace | Frank |
| 8 | Heidi | Frank |
| 9 | Ivan | Frank |
| 10 | Judy | null |
| 11 | Kevin | Judy |
| 12 | Leo | null |
| 13 | Mallory | Leo |
| 14 | Nina | Leo |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT e.emp_id, e.name, m.name AS manager
    FROM employees e
    LEFT JOIN employees m ON e.manager_id = m.emp_id
    ORDER BY e.emp_id
    """
spark.sql(q).show()
q = """
    SELECT m.emp_id, m.name, COUNT(e.emp_id) AS reports
    FROM employees m
    LEFT JOIN employees e ON e.manager_id = m.emp_id
    GROUP BY m.emp_id, m.name
    HAVING COUNT(e.emp_id) = 0
    ORDER BY m.emp_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `LEFT JOIN` keeps the rows whose `manager_id` is `NULL`, which an inner join would drop. `COUNT(e.emp_id)`
counts the real matches, not the `NULL` placeholder row. To walk more than one level, use a recursive CTE.

</details>

### Q13. Customers with no orders (anti-join)

**Difficulty:** Easy · about 5 min

Find customers who have never placed an order, regardless of order status. Return `customer_id` and `name`, ordered by customer ID. Multiple orders for another customer must not affect membership in this result.

**Sample input**

**`customers`**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 2 | Beta LLC |
| 3 | Cyan Inc |
| 4 | Delta Co |
| 5 | Echo Ltd |
| 6 | Foxtrot GmbH |
| 7 | Gamma SA |
| 8 | Hotel BV |

**`orders`**

| order_id | customer_id |
| --- | --- |
| 1 | 1 |
| 2 | 1 |
| 3 | 1 |
| 4 | 2 |
| 5 | 2 |
| 6 | 2 |
| 7 | 3 |
| 8 | 3 |
| 9 | 4 |
| 10 | 4 |
| 11 | 5 |
| 12 | 5 |
| 13 | 6 |
| 14 | 6 |

**Expected output**

| customer_id | name |
| --- | --- |
| 7 | Gamma SA |
| 8 | Hotel BV |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT c.customer_id, c.name
    FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
spark.sql(q).show()
q = """
    SELECT c.customer_id, c.name
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    WHERE o.order_id IS NULL
    ORDER BY c.customer_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** An **anti-join** returns the left rows that have no match on the right. `NOT EXISTS` is the safest form.
`LEFT JOIN ... IS NULL` is equivalent when the column tested for `NULL` is never `NULL` in a real match; `order_id`, the
key of `orders`, qualifies. `NOT IN` is the dangerous one: a single `NULL` in the subquery makes it return no rows (Q15).

</details>

### Q14. Order counts and totals per customer, including customers with none

**Difficulty:** Medium · about 10 min

Return one row for every customer with `customer_id`, `name`, `order_count` and `total_amount`. Count all order statuses. A customer with no orders must have count 0 and amount 0, not a count of 1 from the outer-join placeholder. Order by customer ID.

**Sample input**

**`customers`**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 2 | Beta LLC |
| 3 | Cyan Inc |
| 4 | Delta Co |
| 5 | Echo Ltd |
| 6 | Foxtrot GmbH |
| 7 | Gamma SA |
| 8 | Hotel BV |

**`orders`**

| order_id | customer_id | amount |
| --- | --- | --- |
| 1 | 1 | 100 |
| 2 | 1 | 200 |
| 3 | 1 | 150 |
| 4 | 2 | 300 |
| 5 | 2 | 250 |
| 6 | 2 | 400 |
| 7 | 3 | 120 |
| 8 | 3 | 180 |
| 9 | 4 | 220 |
| 10 | 4 | 320 |
| 11 | 5 | 500 |
| 12 | 5 | 450 |
| 13 | 6 | 275 |
| 14 | 6 | 325 |

**Expected output**

| customer_id | name | order_count | total_amount |
| --- | --- | --- | --- |
| 1 | Acme Corp | 3 | 450 |
| 2 | Beta LLC | 3 | 950 |
| 3 | Cyan Inc | 2 | 300 |
| 4 | Delta Co | 2 | 540 |
| 5 | Echo Ltd | 2 | 950 |
| 6 | Foxtrot GmbH | 2 | 600 |
| 7 | Gamma SA | 0 | 0 |
| 8 | Hotel BV | 0 | 0 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT c.customer_id, c.name,
           COUNT(o.order_id) AS order_count,
           COALESCE(SUM(o.amount), 0) AS total_amount
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY c.customer_id
    """
spark.sql(q).show()
q = """
    SELECT c.customer_id, COUNT(*) AS rows_seen
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id
    HAVING COUNT(*) = 1
    ORDER BY c.customer_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `COUNT(column)` skips `NULL`, `COUNT(*)` does not. `SUM` over zero rows is `NULL`, so wrap it in
`COALESCE`. A predicate on the right-hand table in `WHERE` quietly turns the outer join into an inner join.

</details>

### Q15. `EXISTS` versus `IN`, and the `NOT IN` trap

**Difficulty:** Medium · about 10 min

Return customers with at least one order using EXISTS and then IN. Return `customer_id` and `name`, once per customer, ordered by ID. Separately, find IDs absent from `tiny_orders` using NOT IN and NOT EXISTS, and explain why the null key changes the NOT IN result.

**Sample input**

**`customers`**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 2 | Beta LLC |
| 3 | Cyan Inc |
| 4 | Delta Co |
| 5 | Echo Ltd |
| 6 | Foxtrot GmbH |
| 7 | Gamma SA |
| 8 | Hotel BV |

**`orders`**

| order_id | customer_id |
| --- | --- |
| 1 | 1 |
| 2 | 1 |
| 3 | 1 |
| 4 | 2 |
| 5 | 2 |
| 6 | 2 |
| 7 | 3 |
| 8 | 3 |
| 9 | 4 |
| 10 | 4 |
| 11 | 5 |
| 12 | 5 |
| 13 | 6 |
| 14 | 6 |

**`tiny_orders`**

| customer_id |
| --- |
| 1 |
| null |

**Expected output**

**Customers with orders**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 2 | Beta LLC |
| 3 | Cyan Inc |
| 4 | Delta Co |
| 5 | Echo Ltd |
| 6 | Foxtrot GmbH |

**Customers with orders using IN**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 2 | Beta LLC |
| 3 | Cyan Inc |
| 4 | Delta Co |
| 5 | Echo Ltd |
| 6 | Foxtrot GmbH |

**NOT IN against tiny_orders**

| customer_id |
| --- |

**NOT EXISTS against tiny_orders**

| customer_id |
| --- |
| 2 |
| 3 |
| 4 |
| 5 |
| 6 |
| 7 |
| 8 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT c.customer_id, c.name FROM customers c
    WHERE EXISTS (SELECT 1 FROM orders o WHERE o.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
spark.sql(q).show()
q = """
    SELECT customer_id, name FROM customers
    WHERE customer_id IN (SELECT customer_id FROM orders)
    ORDER BY customer_id
    """
spark.sql(q).show()
q = "SELECT customer_id FROM customers WHERE customer_id NOT IN (SELECT customer_id FROM tiny_orders)"
spark.sql(q).show()
q = """
    SELECT c.customer_id FROM customers c
    WHERE NOT EXISTS (SELECT 1 FROM tiny_orders t WHERE t.customer_id = c.customer_id)
    ORDER BY c.customer_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `EXISTS` is a **semi-join**: it keeps a left row if at least one right row matches, and never
duplicates it. `NOT IN` returns nothing here because `tiny_orders` contains a `NULL`, while `NOT EXISTS` is
`NULL`-safe; [SQL 3: NULL handling](sql-03-null-handling.ipynb), section 6, works through the three-valued logic.
Optimisers often produce the same plan for `IN` and `EXISTS`, so `NULL` safety and readability decide the choice.

</details>

## Part E - Aggregation, conditional counts and medians

Conditional aggregation turns categories into columns. Medians need an interpolation rule, and the rule matters when the
row count is even.

### Q16. Employees paid above their department average

**Difficulty:** Medium · about 10 min

Return employees whose salary is strictly greater than the average salary in their own department. Output `name`, `dept_id` and `salary`, ordered by department, salary descending, then name. Calculate the average using every employee in the department, including those who will be filtered out.

**Sample input**

**`employees`**

| emp_id | name | dept_id | salary |
| --- | --- | --- | --- |
| 1 | Alice | 1 | 150000 |
| 6 | Frank | 2 | 130000 |
| 10 | Judy | 3 | 105000 |
| 12 | Leo | 4 | 90000 |
| 7 | Grace | 2 | 98000 |
| 8 | Heidi | 2 | 87000 |
| 9 | Ivan | 2 | 87000 |
| 11 | Kevin | 3 | 88000 |
| 2 | Bob | 1 | 120000 |
| 3 | Carol | 1 | 120000 |
| 13 | Mallory | 4 | 72000 |
| 14 | Nina | 4 | 68000 |
| 4 | Dave | 1 | 95000 |
| 5 | Eve | 1 | 90000 |

**Expected output**

| name | dept_id | salary |
| --- | --- | --- |
| Alice | 1 | 150000 |
| Bob | 1 | 120000 |
| Carol | 1 | 120000 |
| Frank | 2 | 130000 |
| Judy | 3 | 105000 |
| Leo | 4 | 90000 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT e.name, e.dept_id, e.salary
    FROM employees e
    WHERE e.salary > (SELECT AVG(salary) FROM employees x WHERE x.dept_id = e.dept_id)
    ORDER BY e.dept_id, e.salary DESC, e.name
    """
spark.sql(q).show()
q = """
    SELECT name, dept_id, salary FROM (
        SELECT name, dept_id, salary, AVG(salary) OVER (PARTITION BY dept_id) AS dept_avg
        FROM employees
    )
    WHERE salary > dept_avg
    ORDER BY dept_id, salary DESC, name
    """
spark.sql(q).show()
q = "SELECT dept_id, ROUND(AVG(salary), 2) AS avg_salary FROM employees GROUP BY dept_id ORDER BY dept_id"
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** The correlated subquery expresses the department-average lookup for each employee; the optimizer may decorrelate it. The window version computes
it once per partition with `AVG(...) OVER (PARTITION BY dept_id)`, and filters in an outer query. "Above" means a
strict `>`: an employee paid exactly the average is not above it. `AVG` returns a `DOUBLE`, so the comparison is exact
for these integer salaries.

</details>

### Q17. Pivot counts by status

**Difficulty:** Medium · about 10 min

Return every customer with `customer_id`, `name`, `completed`, `pending` and `cancelled`: one count column per order status. Customers without orders get zeros in all three columns. Order by customer ID. Do not discard them when constructing the pivot.

**Sample input**

**`customers`**

| customer_id | name |
| --- | --- |
| 1 | Acme Corp |
| 2 | Beta LLC |
| 3 | Cyan Inc |
| 4 | Delta Co |
| 5 | Echo Ltd |
| 6 | Foxtrot GmbH |
| 7 | Gamma SA |
| 8 | Hotel BV |

**`orders`**

| order_id | customer_id | status |
| --- | --- | --- |
| 1 | 1 | completed |
| 2 | 1 | completed |
| 3 | 1 | completed |
| 4 | 2 | completed |
| 5 | 2 | cancelled |
| 6 | 2 | completed |
| 7 | 3 | completed |
| 8 | 3 | completed |
| 9 | 4 | pending |
| 10 | 4 | completed |
| 11 | 5 | completed |
| 12 | 5 | completed |
| 13 | 6 | completed |
| 14 | 6 | cancelled |

**Expected output**

| customer_id | name | completed | pending | cancelled |
| --- | --- | --- | --- | --- |
| 1 | Acme Corp | 3 | 0 | 0 |
| 2 | Beta LLC | 2 | 0 | 1 |
| 3 | Cyan Inc | 2 | 0 | 0 |
| 4 | Delta Co | 1 | 1 | 0 |
| 5 | Echo Ltd | 2 | 0 | 0 |
| 6 | Foxtrot GmbH | 1 | 0 | 1 |
| 7 | Gamma SA | 0 | 0 | 0 |
| 8 | Hotel BV | 0 | 0 | 0 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT c.customer_id, c.name,
           COUNT(CASE WHEN o.status = 'completed' THEN 1 END) AS completed,
           COUNT(CASE WHEN o.status = 'pending'   THEN 1 END) AS pending,
           COUNT(CASE WHEN o.status = 'cancelled' THEN 1 END) AS cancelled
    FROM customers c
    LEFT JOIN orders o ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.name
    ORDER BY c.customer_id
    """
spark.sql(q).show()
q = """
    SELECT customer_id,
           COALESCE(completed, 0) AS completed,
           COALESCE(pending, 0)   AS pending,
           COALESCE(cancelled, 0) AS cancelled
    FROM (
        SELECT * FROM (SELECT customer_id, status FROM orders)
        PIVOT (COUNT(*) FOR status IN ('completed' AS completed, 'pending' AS pending, 'cancelled' AS cancelled))
    )
    ORDER BY customer_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `COUNT(CASE ... THEN 1 END)` is the portable form: no `ELSE` gives `NULL`, which `COUNT` ignores. With `SUM`,
you need `ELSE 0`. Spark's `PIVOT` produces `NULL` for missing combinations and drops groups with no rows.

</details>

### Q18. Median salary

**Difficulty:** Medium · about 15 min

Return the median salary as one row named `median`. For an even number of employees, average the two middle values after sorting; for an odd number, use the middle value. Show a percentile-function solution and a window-function alternative.

**Sample input**

**`employees`**

| emp_id | name | salary |
| --- | --- | --- |
| 1 | Alice | 150000 |
| 6 | Frank | 130000 |
| 10 | Judy | 105000 |
| 12 | Leo | 90000 |
| 7 | Grace | 98000 |
| 8 | Heidi | 87000 |
| 9 | Ivan | 87000 |
| 11 | Kevin | 88000 |
| 2 | Bob | 120000 |
| 3 | Carol | 120000 |
| 13 | Mallory | 72000 |
| 14 | Nina | 68000 |
| 4 | Dave | 95000 |
| 5 | Eve | 90000 |

**Expected output**

| median |
| --- |
| 92500.0 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = "SELECT percentile_cont(0.5) WITHIN GROUP (ORDER BY salary) AS median FROM employees"
spark.sql(q).show()
q = """
    WITH ranked AS (
        SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS rn, COUNT(*) OVER () AS n FROM employees
    )
    SELECT AVG(salary) FROM ranked
    WHERE rn IN (FLOOR((n + 1) / 2), FLOOR((n + 2) / 2))
    """
spark.sql(q).show()
q = """
    SELECT salary FROM (
        SELECT salary, ROW_NUMBER() OVER (ORDER BY salary) AS rn, COUNT(*) OVER () AS n FROM employees
    )
    WHERE rn IN (FLOOR((n + 1) / 2), FLOOR((n + 2) / 2))
    ORDER BY salary
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** `percentile_cont` interpolates between the two middle values, so an even count gives their mean. The window
version does the same by hand and is what you reach for when the engine lacks a percentile function.

</details>

## Part F - Time, sequences and cohorts

Gaps and islands, sessionization and cohort retention all reduce to "compare each row with the previous row in order",
which is `LAG` plus a cumulative `SUM`.

### Q19. Consecutive-day streaks and the longest streak per user

**Difficulty:** Hard · about 20 min

Find every consecutive-calendar-day streak for every user in `events`, counting a date once even if it has several events. Return `user_id`, `start_day`, `end_day` and `streak_len`, ordered by user and start day. Then return each user's longest streak as `user_id` and `longest`. Any event type counts as activity.

**Sample input**

**`events`**

| event_id | user_id | event_ts |
| --- | --- | --- |
| 1 | 1 | 2023-01-01 09:00:00 |
| 2 | 1 | 2023-01-02 09:05:00 |
| 3 | 1 | 2023-01-03 08:55:00 |
| 4 | 1 | 2023-01-05 09:00:00 |
| 5 | 1 | 2023-01-06 09:10:00 |
| 6 | 2 | 2023-01-01 10:00:00 |
| 7 | 2 | 2023-01-02 10:00:00 |
| 8 | 2 | 2023-01-04 10:00:00 |
| 9 | 3 | 2023-02-01 10:00:00 |
| 10 | 3 | 2023-02-01 10:05:00 |
| 11 | 3 | 2023-02-01 10:25:00 |
| 12 | 3 | 2023-02-01 11:30:00 |
| 13 | 3 | 2023-02-01 11:45:00 |
| 14 | 4 | 2023-02-01 10:00:00 |
| 15 | 4 | 2023-02-01 10:20:00 |

**Expected output**

**All streaks**

| user_id | start_day | end_day | streak_len |
| --- | --- | --- | --- |
| 1 | 2023-01-01 | 2023-01-03 | 3 |
| 1 | 2023-01-05 | 2023-01-06 | 2 |
| 2 | 2023-01-01 | 2023-01-02 | 2 |
| 2 | 2023-01-04 | 2023-01-04 | 1 |
| 3 | 2023-02-01 | 2023-02-01 | 1 |
| 4 | 2023-02-01 | 2023-02-01 | 1 |

**Longest streak per user**

| user_id | longest |
| --- | --- |
| 1 | 3 |
| 2 | 2 |
| 3 | 1 |
| 4 | 1 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    WITH daily AS (
        SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
    ),
    grouped AS (
        SELECT user_id, d,
               date_sub(d, CAST(ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) AS INT)) AS island
        FROM daily
    )
    SELECT user_id, MIN(d) AS start_day, MAX(d) AS end_day, COUNT(*) AS streak_len
    FROM grouped
    GROUP BY user_id, island
    ORDER BY user_id, start_day
    """
spark.sql(q).show()
q = """
    WITH daily AS (
        SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
    ),
    grouped AS (
        SELECT user_id, d,
               date_sub(d, CAST(ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) AS INT)) AS island
        FROM daily
    )
    SELECT user_id, MAX(streak_len) AS longest FROM (
        SELECT user_id, island, COUNT(*) AS streak_len FROM grouped GROUP BY user_id, island
    )
    GROUP BY user_id
    ORDER BY user_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** Deduplicate to one row per day first, because several events on one day count as one day. Then number the
days per user. Within a run of consecutive days, the date and the row number both rise by one each step, so the date
minus the row number is constant. Worked by hand for user 1:

| day | row number | day minus row number |
| --- | --- | --- |
| 2023-01-01 | 1 | 2022-12-31 |
| 2023-01-02 | 2 | 2022-12-31 |
| 2023-01-03 | 3 | 2022-12-31 |
| 2023-01-05 | 4 | 2023-01-01 |
| 2023-01-06 | 5 | 2023-01-01 |

Grouping by that constant, called the **island** key, finds each run. `date_sub` needs an integer, so the row number
is cast.

</details>

### Q20. Sessions: a new session after 30 minutes of inactivity

**Difficulty:** Hard · about 20 min

Count sessions per user. Each user's first event starts a session; a later event starts another only when the gap from that user's previous event is greater than 30 minutes. A gap of exactly 30 minutes stays in the session. Order events by timestamp and event ID. Return `user_id` and `sessions`, ordered by user.

**Sample input**

**`events`**

| event_id | user_id | event_ts |
| --- | --- | --- |
| 1 | 1 | 2023-01-01 09:00:00 |
| 2 | 1 | 2023-01-02 09:05:00 |
| 3 | 1 | 2023-01-03 08:55:00 |
| 4 | 1 | 2023-01-05 09:00:00 |
| 5 | 1 | 2023-01-06 09:10:00 |
| 6 | 2 | 2023-01-01 10:00:00 |
| 7 | 2 | 2023-01-02 10:00:00 |
| 8 | 2 | 2023-01-04 10:00:00 |
| 9 | 3 | 2023-02-01 10:00:00 |
| 10 | 3 | 2023-02-01 10:05:00 |
| 11 | 3 | 2023-02-01 10:25:00 |
| 12 | 3 | 2023-02-01 11:30:00 |
| 13 | 3 | 2023-02-01 11:45:00 |
| 14 | 4 | 2023-02-01 10:00:00 |
| 15 | 4 | 2023-02-01 10:20:00 |

**Expected output**

| user_id | sessions |
| --- | --- |
| 1 | 5 |
| 2 | 3 |
| 3 | 2 |
| 4 | 1 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    WITH ordered AS (
        SELECT event_id, user_id, event_ts,
               LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
        FROM events
    ),
    flagged AS (
        SELECT event_id, user_id, event_ts,
               CASE WHEN prev_ts IS NULL
                      OR (unix_timestamp(event_ts) - unix_timestamp(prev_ts)) / 60.0 > 30
                    THEN 1 ELSE 0 END AS is_new_session
        FROM ordered
    ),
    numbered AS (
        SELECT user_id,
               SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_id
        FROM flagged
    )
    SELECT user_id, MAX(session_id) AS sessions FROM numbered GROUP BY user_id ORDER BY user_id
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** Flag the boundaries, then take a running sum of the flags. The first event per user has no previous
timestamp and must start a session explicitly. Tie-break on `event_id` so the order is deterministic.

</details>

### Q21. Cohort retention by month

**Difficulty:** Hard · about 20 min

Assign each ordering customer to the calendar month of their first order. For each cohort and active month, count distinct active customers. Any order status counts as activity, including cancelled and pending orders. Return `cohort_month`, `activity_month` and `active_customers`, ordered by the two month columns. Customers with no orders are excluded; multiple orders in one month count once.

**Sample input**

**`orders`**

| order_id | customer_id | status | amount | order_date |
| --- | --- | --- | --- | --- |
| 1 | 1 | completed | 100 | 2023-01-05 |
| 2 | 1 | completed | 200 | 2023-02-10 |
| 3 | 1 | completed | 150 | 2023-03-15 |
| 4 | 2 | completed | 300 | 2023-01-20 |
| 5 | 2 | cancelled | 250 | 2023-02-05 |
| 6 | 2 | completed | 400 | 2023-03-20 |
| 7 | 3 | completed | 120 | 2023-01-25 |
| 8 | 3 | completed | 180 | 2023-02-28 |
| 9 | 4 | pending | 220 | 2023-02-14 |
| 10 | 4 | completed | 320 | 2023-03-30 |
| 11 | 5 | completed | 500 | 2023-01-02 |
| 12 | 5 | completed | 450 | 2023-03-05 |
| 13 | 6 | completed | 275 | 2023-02-18 |
| 14 | 6 | cancelled | 325 | 2023-03-25 |

**Expected output**

| cohort_month | activity_month | active_customers |
| --- | --- | --- |
| 2023-01 | 2023-01 | 4 |
| 2023-01 | 2023-02 | 3 |
| 2023-01 | 2023-03 | 3 |
| 2023-02 | 2023-02 | 2 |
| 2023-02 | 2023-03 | 2 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    WITH first_order AS (
        SELECT customer_id, MIN(date_format(order_date, 'yyyy-MM')) AS cohort_month
        FROM orders GROUP BY customer_id
    ),
    activity AS (
        SELECT DISTINCT customer_id, date_format(order_date, 'yyyy-MM') AS activity_month FROM orders
    )
    SELECT f.cohort_month, a.activity_month, COUNT(DISTINCT a.customer_id) AS active_customers
    FROM first_order f
    JOIN activity a ON a.customer_id = f.customer_id
    GROUP BY f.cohort_month, a.activity_month
    ORDER BY f.cohort_month, a.activity_month
    """
spark.sql(q).show()
q = """
    WITH first_order AS (
        SELECT customer_id, MIN(date_format(order_date, 'yyyy-MM')) AS cohort_month
        FROM orders GROUP BY customer_id
    )
    SELECT cohort_month, COUNT(*) AS cohort_size FROM first_order GROUP BY cohort_month ORDER BY cohort_month
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** Three steps: each customer's first period, the distinct active (customer, period) pairs, and the join.
`COUNT(DISTINCT customer_id)` stops a customer with several orders in one month from inflating retention.

</details>

### Q22. Cumulative distinct customers by date

**Difficulty:** Hard · about 15 min

For every distinct date in `orders`, count customers with at least one order on or before that date. Include every order status. Return `order_date` and `cum_customers`, ordered by date. Repeated orders by an existing customer must not increment the count; include dates when no new customer appeared.

**Sample input**

**`orders`**

| order_id | customer_id | order_date |
| --- | --- | --- |
| 1 | 1 | 2023-01-05 |
| 2 | 1 | 2023-02-10 |
| 3 | 1 | 2023-03-15 |
| 4 | 2 | 2023-01-20 |
| 5 | 2 | 2023-02-05 |
| 6 | 2 | 2023-03-20 |
| 7 | 3 | 2023-01-25 |
| 8 | 3 | 2023-02-28 |
| 9 | 4 | 2023-02-14 |
| 10 | 4 | 2023-03-30 |
| 11 | 5 | 2023-01-02 |
| 12 | 5 | 2023-03-05 |
| 13 | 6 | 2023-02-18 |
| 14 | 6 | 2023-03-25 |

**Expected output**

| order_date | cum_customers |
| --- | --- |
| 2023-01-02 | 1 |
| 2023-01-05 | 2 |
| 2023-01-20 | 3 |
| 2023-01-25 | 4 |
| 2023-02-05 | 4 |
| 2023-02-10 | 4 |
| 2023-02-14 | 5 |
| 2023-02-18 | 6 |
| 2023-02-28 | 6 |
| 2023-03-05 | 6 |
| 2023-03-15 | 6 |
| 2023-03-20 | 6 |
| 2023-03-25 | 6 |
| 2023-03-30 | 6 |

In [ ]:
# Write your query here. Run it with spark.sql(query).show().

In [ ]:
q = """
    SELECT o.order_date,
           (SELECT COUNT(DISTINCT o2.customer_id) FROM orders o2 WHERE o2.order_date <= o.order_date) AS cum_customers
    FROM (SELECT DISTINCT order_date FROM orders) o
    ORDER BY o.order_date
    """
spark.sql(q).show()
q = """
    WITH first_order AS (
        SELECT customer_id, MIN(order_date) AS first_date FROM orders GROUP BY customer_id
    )
    SELECT first_date, SUM(COUNT(*)) OVER (ORDER BY first_date) AS cum_customers
    FROM first_order
    GROUP BY first_date
    ORDER BY first_date
    """
spark.sql(q).show()

<details>
<summary>Why this works</summary>

**Approach.** The correlated count recounts history for every date. The scalable version counts each customer once, at their
first order, and takes a running sum. Summing `COUNT(DISTINCT)` over raw periods double-counts customers who order in several.

</details>

## Concept questions

The concept questions have their own chapter, with short answers and runnable demos:
[Interview concepts](sql-14-interview-concepts.ipynb).

## Where to go next

Re-do the problems you got wrong, then read the chapter behind each part:
[window functions](sql-08-window-functions.ipynb) for Parts A and C,
[deduplication](sql-09-deduplication.ipynb) for Part B, [joins](sql-02-joins.ipynb) for Part D,
[aggregations](sql-05-aggregations-and-ctes.ipynb) for Part E and [time series](sql-10-time-series.ipynb) for Part F.